# RAG Veltis — Qdrant dense + BM25 sparse + hybrid RRF
**Cours reconstitué — 14 octobre 2026** · Suite de `demo_qdrant.ipynb` et de l’exercice Veltis du 12 octobre.

**Objectif :** reprendre les 84 passages historiques (chunker cible 544 tokens), conserver leurs vecteurs BGE-M3, ajouter un index lexical BM25, puis comparer **dense / sparse / hybride RRF**. La variante 100 tokens est optionnelle et ne doit être utilisée que si ses fichiers existent réellement.

**Consigne :** exécuter **une cellule à la fois** et contrôler chaque sortie. Ne pas supprimer les anciennes collections. Les nouvelles collections portent `hybrid` dans leur nom.

**Attention :** ce notebook contient des opérations qui écrivent dans Qdrant (création de collection / upsert). L'API locale utilise la clé de démonstration du cours, à remplacer par une configuration sûre pour tout déploiement public.

## Cellule 1 — Imports, chemins, paramètres (aucun réseau)
Le notebook charge les fichiers enregistrés le 12 octobre, sans relancer Docling ni générer les embeddings historiques.

In [1]:
# CELLULE 1 — IMPORTS ET CONFIGURATION LOCALE
from pathlib import Path
import json
import uuid
import numpy as np
from openai import OpenAI
from qdrant_client import QdrantClient, models

ROOT = Path('/home/bm/formation/RAG/cours-rag/drive-download-20261008T064617Z-1-001/rag/demo')
EXPORT = ROOT / 'export_folder'
OLD_EMBEDDINGS = EXPORT / 'veltis_embeddings.npz'
OLD_CHUNKS = EXPORT / 'veltis_chunks.json'
NEW_EMBEDDINGS = EXPORT / 'veltis_embeddings_100.npz'
NEW_CHUNKS = EXPORT / 'veltis_chunks_100.json'

QDRANT_URL = 'http://127.0.0.1:6333'
EMBEDDER_URL = 'http://127.0.0.1:8001/v1'
DENSE_MODEL = 'bge-m3'
SPARSE_MODEL = 'qdrant/bm25'
COLL_544 = 'veltis_hybrid_544_20261014'
COLL_100 = 'veltis_hybrid_100_20261014'

# Configuration du tokenizer BM25 du cours; la même doit servir à l'indexation ET aux questions.
BM25_OPTIONS = {'language': 'french', 'ascii_folding': True, 'avg_len': 50}

assert OLD_EMBEDDINGS.exists() and OLD_CHUNKS.exists(), 'Fichiers 544 absents sur la Z4'
print('Corpus historique disponible :', OLD_CHUNKS.name, OLD_EMBEDDINGS.name)
print('Variante 100 disponible :', NEW_EMBEDDINGS.exists() and NEW_CHUNKS.exists())
print('Collections visées :', COLL_544, '/', COLL_100)

Corpus historique disponible : veltis_chunks.json veltis_embeddings.npz
Variante 100 disponible : True
Collections visées : veltis_hybrid_544_20261014 / veltis_hybrid_100_20261014


## Cellule 2 — Connexion à Qdrant et au serveur BGE-M3
Les services sont **distincts** : Qdrant stocke et classe les vecteurs; vLLM calcule les embeddings de la question. Le serveur BGE-M3 n’est pas nécessaire pour *importer* les 84 vecteurs sauvegardés, mais il le sera pour tester les questions.

In [2]:
# CELLULE 2 — CONNEXION QDRANT (AUTHENTIFICATION DU COURS)
# Clé locale de démonstration reprise de demo_qdrant.ipynb.
# Ne pas publier ce notebook avec une vraie clé de production.
qclient = QdrantClient(url=QDRANT_URL, api_key='1234', timeout=30)
existing = qclient.get_collections().collections
print('Qdrant OK. Collections existantes :', [x.name for x in existing])

# Le client est défini maintenant; il fera appel au serveur BGE-M3 seulement à la requête.
embedder = OpenAI(base_url=EMBEDDER_URL, api_key='not-needed', timeout=120.0)

Qdrant OK. Collections existantes : ['demo_collection', 'veltis_dense_100', 'veltis_dense_544', 'veltis_hybrid_100_20261014', 'veltis_hybrid_544_20261014']


/tmp/ipykernel_3502788/964658924.py:4: UserWarning: Api key is used with an insecure connection.
  qclient = QdrantClient(url=QDRANT_URL, api_key='1234', timeout=30)


## Cellule 3 — Charger et contrôler les 84 chunks historiques
Chaque **ligne i** de la matrice correspond à la **métadonnée i** du fichier JSON. L’ordre est indispensable pour préserver les sources.

In [3]:
# CELLULE 3 — CHARGEMENT DES DONNÉES HISTORIQUES (544)
with np.load(OLD_EMBEDDINGS) as archive:
    vecs_544 = archive['embeddings']
with OLD_CHUNKS.open(encoding='utf-8') as fh:
    chunks_544 = json.load(fh)

assert vecs_544.shape == (84, 1024), vecs_544.shape
assert len(chunks_544) == len(vecs_544)
assert np.isfinite(vecs_544).all()
print('Matrice dense :', vecs_544.shape)
print('Métadonnées :', len(chunks_544))
print('Chunk 69 :', chunks_544[69]['source'], '|', chunks_544[69]['text'][:85])

Matrice dense : (84, 1024)
Métadonnées : 84
Chunk 69 : pv_conseil_administration_2026-03-12_scan.pdf | 2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de


## Cellule 4 — Créer une nouvelle collection hybride (dense + sparse)
`size=1024` correspond à BGE-M3. `COSINE` correspond à notre exercice NumPy. `modifier=IDF` configure le poids inverse de la fréquence documentaire du vecteur sparse. **La création d’un champ sparse ne génère pas elle-même le vecteur BM25**.

In [4]:
# CELLULE 4 — CRÉER UNE COLLECTION HYBRIDE SANS ÉCRASER L'EXISTANT
if not qclient.collection_exists(COLL_544):
    qclient.create_collection(
        collection_name=COLL_544,
        vectors_config={
            'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE)
        },
        sparse_vectors_config={
            'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)
        }
    )
    print('Créée :', COLL_544)
else:
    print('Collection déjà présente :', COLL_544)

info = qclient.get_collection(COLL_544)
assert 'dense' in info.config.params.vectors
assert 'sparse' in info.config.params.sparse_vectors
print('Vecteurs nommés : dense + sparse | Nombre de points :', info.points_count)

Collection déjà présente : veltis_hybrid_544_20261014
Vecteurs nommés : dense + sparse | Nombre de points : 84


## Cellule 5 — Préparer les points avec les **deux** représentations
Comme dans `demo_qdrant.ipynb`, chaque point contient un vecteur `dense` et un `models.Document(..., model="qdrant/bm25")`. Qdrant génère la représentation sparse à partir du texte lors de l’insertion **si le serveur prend en charge cette fonctionnalité**.

`uuid5` produit des identifiants stables : un nouvel upsert ne duplique pas les mêmes chunks.

In [5]:
# CELLULE 5 — CONSTRUIRE LES POINTS 544 EN MÉMOIRE (PAS ENCORE D'UPSERT)
def build_points(chunks, vectors, profile):
    assert len(chunks) == len(vectors)
    result = []
    for i, (chunk, vec) in enumerate(zip(chunks, vectors)):
        source = chunk['source']
        text = chunk['text']
        stable_id = str(uuid.uuid5(uuid.NAMESPACE_URL, f'veltis/{profile}/{i}/{source}'))
        result.append(models.PointStruct(
            id=stable_id,
            vector={
                'dense': vec.astype(np.float32).tolist(),
                'sparse': models.Document(text=text, model=SPARSE_MODEL, options=BM25_OPTIONS),
            },
            payload={
                'dataset': 'veltis', 'chunk_profile': profile,
                'file_name': source, 'global_index': i, 'text': text,
            },
        ))
    return result

points_544 = build_points(chunks_544, vecs_544, '544')
print('Points préparés :', len(points_544))
print('Exemple source :', points_544[69].payload['file_name'])
print('Champs vecteurs :', list(points_544[0].vector))

Points préparés : 84
Exemple source : pv_conseil_administration_2026-03-12_scan.pdf
Champs vecteurs : ['dense', 'sparse']


## Cellule 6 — Insérer les points 544 dans Qdrant
**Écriture effective** dans Qdrant. Insertion par lots pour réduire les risques d’échec. Si l’instance Qdrant ne prend pas en charge l’inférence BM25 côté serveur, cette cellule peut renvoyer une erreur : il faudra alors vérifier sa version et son mode de déploiement plutôt que prétendre que le sparse a été indexé.

In [6]:
# CELLULE 6 — UPSERT DENSE + SPARSE / 544 TOKENS
BATCH_SIZE = 16
for start in range(0, len(points_544), BATCH_SIZE):
    batch = points_544[start:start + BATCH_SIZE]
    result = qclient.upsert(collection_name=COLL_544, points=batch, wait=True)
    print(f'Lot {start + 1} à {start + len(batch)} : {result.status}')

count = qclient.count(collection_name=COLL_544, exact=True).count
print('Points dans la collection :', count)
assert count == len(points_544), f'{count} points au lieu de {len(points_544)}'
print('IMPORT HYBRIDE 544 : SUCCESS')

Lot 1 à 16 : completed
Lot 17 à 32 : completed
Lot 33 à 48 : completed
Lot 49 à 64 : completed
Lot 65 à 80 : completed
Lot 81 à 84 : completed
Points dans la collection : 84
IMPORT HYBRIDE 544 : SUCCESS


## Cellule 7 — Charger la variante 100 tokens, **si elle existe**
On ne recrée pas les chunks : on utilise uniquement les fichiers `veltis_embeddings_100.npz` et `veltis_chunks_100.json` déjà produits dans notre autre exercice. Si absents, cette étape indique explicitement qu’il faut d’abord terminer l’exercice 100 tokens.

In [7]:
# CELLULE 7 — VARIANTE 100 : CHARGEMENT CONDITIONNEL
available_100 = NEW_EMBEDDINGS.is_file() and NEW_CHUNKS.is_file()
vecs_100, chunks_100 = None, None
if not available_100:
    print('Variante 100 non disponible. Ne pas exécuter les cellules 8–9.')
else:
    with np.load(NEW_EMBEDDINGS) as archive:
        vecs_100 = archive['embeddings']
    with NEW_CHUNKS.open(encoding='utf-8') as fh:
        chunks_100 = json.load(fh)
    assert vecs_100.ndim == 2 and vecs_100.shape[1] == 1024
    assert len(chunks_100) == len(vecs_100)
    assert np.isfinite(vecs_100).all()
    print('Variante 100 :', vecs_100.shape, 'chunks :', len(chunks_100))

Variante 100 : (129, 1024) chunks : 129


## Cellule 8 — Créer la collection hybride 100 (optionnel)

In [8]:
# CELLULE 8 — CRÉATION COLLECTION 100 (SI LES FICHIERS EXISTENT)
if not available_100:
    print('Cellule ignorée : variante 100 absente.')
elif not qclient.collection_exists(COLL_100):
    qclient.create_collection(
        collection_name=COLL_100,
        vectors_config={'dense': models.VectorParams(size=1024, distance=models.Distance.COSINE)},
        sparse_vectors_config={'sparse': models.SparseVectorParams(modifier=models.Modifier.IDF)}
    )
    print('Créée :', COLL_100)
else:
    info100 = qclient.get_collection(COLL_100)
    assert 'dense' in info100.config.params.vectors
    assert 'sparse' in info100.config.params.sparse_vectors
    print('Déjà présente :', COLL_100)

Déjà présente : veltis_hybrid_100_20261014


## Cellule 9 — Insérer la variante 100 (optionnel)

In [9]:
# CELLULE 9 — UPSERT VARIANTE 100
if not available_100:
    print('Cellule ignorée : variante 100 absente.')
else:
    points_100 = build_points(chunks_100, vecs_100, '100')
    for start in range(0, len(points_100), BATCH_SIZE):
        batch = points_100[start:start + BATCH_SIZE]
        result = qclient.upsert(collection_name=COLL_100, points=batch, wait=True)
        print(f'Lot 100 tokens {start + 1}–{start + len(batch)} : {result.status}')
    count100 = qclient.count(collection_name=COLL_100, exact=True).count
    assert count100 == len(points_100)
    print('IMPORT HYBRIDE 100 : SUCCESS', count100)

Lot 100 tokens 1–16 : completed
Lot 100 tokens 17–32 : completed
Lot 100 tokens 33–48 : completed
Lot 100 tokens 49–64 : completed
Lot 100 tokens 65–80 : completed
Lot 100 tokens 81–96 : completed
Lot 100 tokens 97–112 : completed
Lot 100 tokens 113–128 : completed
Lot 100 tokens 129–129 : completed
IMPORT HYBRIDE 100 : SUCCESS 129


## Cellule 10 — Définir la question et créer son embedding dense
La question est transformée en vecteur BGE-M3, exactement comme dans notre exercice NumPy. Le **sparse** sera calculé par Qdrant à partir du texte de la même question.

In [10]:
# CELLULE 10 — QUESTION ET VECTEUR DENSE
question = 'Quel dividende sera proposé aux actionnaires ?'
response = embedder.embeddings.create(model=DENSE_MODEL, input=[question])
question_embedded = np.asarray(response.data[0].embedding, dtype=np.float32).tolist()
assert len(question_embedded) == 1024
print('Question :', question)
print('Dimensions du vecteur question :', len(question_embedded))

Question : Quel dividende sera proposé aux actionnaires ?
Dimensions du vecteur question : 1024


## Cellule 11 — Dense search uniquement
`using="dense"` sélectionne nos vecteurs BGE-M3; Qdrant calcule la proximité cosinus. Cette étape fournit un classement de référence.

In [11]:
# CELLULE 11 — RÉSULTATS DENSE SEULEMENT
results_dense = qclient.query_points(
    collection_name=COLL_544, query=question_embedded,
    using='dense', limit=5, with_payload=True
)
for rank, p in enumerate(results_dense.points, 1):
    print(rank, round(p.score, 4), p.payload.get('file_name'),
          '|', p.payload.get('text', '')[:180])

1 0.633 pv_conseil_administration_2026-03-12_scan.pdf | 2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et 
2 0.4747 rapport_annuel_2025.pdf | 3. Chiffres clés
Chiffre d'affaires (M€), 2024 = 31,2. Chiffre d'affaires (M€), 2025 = 38,4. Chiffre d'affaires (M€), Évolution = +23,1 %. EBITDA (M€), 2024 = 3,1. EBITDA (M€), 202
3 0.4323 pv_conseil_administration_2026-03-12_scan.pdf | 4. Politique de rémunération variable
t     o       des ressources humaines d'introduire une prime collective liée aux objectifs de qualité. Le sujet sera approfondi et soumis au v
4 0.429 grille_salariale_2026.pdf | 2. Avantages complémentaires
Titres-repas, Bénéficiaires = Tout le personnel. Titres-repas, Montant ou conditions = 8 € par jour travaillé. Indemnité vélo, Bénéficiaires = Trajets 
5 0.4289 rapport_annuel_2025.pdf | 6. Détail des investissements 2025
 2025-06, Investisseme

## Cellule 12 — Sparse / BM25 uniquement
La recherche BM25 privilégie les termes discriminants du texte. `using="sparse"` doit correspondre **exactement** au nom du vecteur sparse configuré dans la collection.

In [12]:
# CELLULE 12 — RÉSULTATS SPARSE / BM25 SEULEMENT
results_sparse = qclient.query_points(
    collection_name=COLL_544,
    query=models.Document(text=question, model=SPARSE_MODEL, options=BM25_OPTIONS),
    using='sparse', limit=5, with_payload=True
)
for rank, p in enumerate(results_sparse.points, 1):
    print(rank, round(p.score, 4), p.payload.get('file_name'),
          '|', p.payload.get('text', '')[:180])

1 8.1006 pv_conseil_administration_2026-03-12_scan.pdf | 2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et 
2 4.973 guide_onboarding.pdf | 4.1 Formation sécurité
Une formation sécurité de 3 heures est obligatoire avant tout accès à l'atelier, quel que soit votre poste. Elle a lieu le mardi ou le jeudi de votre premièr
3 3.3059 intranet_actualites.html | Actualités
Partenariat avec CityLoc pour la location longue durée
Veltis et le loueur CityLoc ont signé un partenariat pour proposer le Veltis Pro L en location longue durée aux co
4 2.9042 newsletter_le_fil_2026-02.pdf | Le saviez-vous ?
Un Veltis Famille parcourt en moyenne 2 300 kilomètres par an, d'après les données anonymisées des vélos connectés de nos clients qui ont accepté de les partager.

5 2.7379 rapport_annuel_2025.pdf | 7. Perspectives 2026
Pour accompagner la croissance, la direction soumettra 

## Cellule 13 — `prefetch` + fusion RRF : le bloc du professeur
`prefetch` constitue deux listes de candidats, chacune limitée à 30 passages : l’une dense, l’autre sparse. La requête principale **fusionne les rangs** avec Reciprocal Rank Fusion (RRF), puis renvoie les 10 premiers passages.

Corrections de transcription : `denese` → `dense` ; `qdrqnt/bm25` → `qdrant/bm25` ; virgule entre les deux `Prefetch` ; ajout de `using="sparse"` ; syntaxe documentée `models.FusionQuery(fusion=models.Fusion.RRF)`. Les scores RRF ne sont **pas** des cosinus.

In [15]:
# CELLULE 13 — HYBRID SEARCH / RRF (BLOC FINAL DU COURS)
results_hybrid = qclient.query_points(
    collection_name=COLL_544,
    prefetch=[
        models.Prefetch(
            query=question_embedded,  # vecteur dense BGE-M3 (1024 dimensions)
            using='dense',           # nom du champ vectoriel dense
            limit=30                 # 30 meilleurs candidats dense
        ),
        models.Prefetch(
            query=models.Document(
                text=question,       # même question, cette fois en texte
                model=SPARSE_MODEL,  # Qdrant convertit le texte en BM25
                options=BM25_OPTIONS
            ),
            using='sparse',          # nom du champ vectoriel sparse
            limit=30                 # 30 meilleurs candidats sparse
        )
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=10,                       # TOP 10 après la fusion
    with_payload=True
)

print('QUESTION :', question)
for rank, point in enumerate(results_hybrid.points, 1):
    print(f"{rank:2d}. RRF={point.score:.5f} | {point.payload.get('file_name')}")
    print('   ', point.payload.get('text', '')[:250])

QUESTION : Quel dividende sera proposé aux actionnaires ?
 1. RRF=1.00000 | pv_conseil_administration_2026-03-12_scan.pdf
    2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et décide de proposer à l'assemblée générale la distribution d'un dividen
 2. RRF=0.33333 | rapport_annuel_2025.pdf
    3. Chiffres clés
Chiffre d'affaires (M€), 2024 = 31,2. Chiffre d'affaires (M€), 2025 = 38,4. Chiffre d'affaires (M€), Évolution = +23,1 %. EBITDA (M€), 2024 = 3,1. EBITDA (M€), 2025 = 4,6. EBITDA (M€), Évolution = +48,4 %. Résultat net (M€), 2024 = 1
 3. RRF=0.33333 | guide_onboarding.pdf
    4.1 Formation sécurité
Une formation sécurité de 3 heures est obligatoire avant tout accès à l'atelier, quel que soit votre poste. Elle a lieu le mardi ou le jeudi de votre première semaine. Les chaussures de sécurité et les lunettes de protection vo
 4. RRF=0.30556 | intranet_actua

## Cellule 14 — Refaire la même requête pour la variante 100 (optionnel)
**Ne pas exécuter avant l’importation de la collection 100.** Une comparaison valable exige les mêmes questions, mêmes modèles et des passages de référence vérifiés, pas simplement le score brut.

In [17]:
result100 = qclient.query_points(
    collection_name=COLL_100,

    prefetch=[
        # Recherche dense : similarité sémantique BGE-M3
        models.Prefetch(
            query=question_embedded,
            using="dense",
            limit=30
        ),

        # Recherche sparse : recherche lexicale BM25
        models.Prefetch(
            query=models.Document(
                text=question,
                model=SPARSE_MODEL,
                options=BM25_OPTIONS
            ),
            using="sparse",
            limit=30
        )
    ],

    # RRF = Reciprocal Rank Fusion
    # Fusion pondérée des résultats Dense + Sparse
    query=models.RrfQuery(
        rrf=models.Rrf(
            k=60,
            weights=[2.0, 1.0]
        )
    ),

    limit=10,
    with_payload=True
)

for rank, point in enumerate(result100.points, 1):
    print(
        rank,
        point.score,
        point.payload.get("file_name"),
        point.payload.get("text", "")[:300]
    )

1 0.03347339 pv_conseil_administration_2026-03-12_scan.pdf 2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et décide de proposer à l'assemblée générale la distribution d'un dividende de 0,80€ par action.
2 0.016666668 rapport_annuel_2025.pdf 3. Chiffres clés

Chiffre d'affaires (M€), 2024 = 31,2. Chiffre d'affaires (M€), 2025 = 38,4. Chiffre d'affaires (M€), Évolution = +23,1 %. EBITDA (M€), 2024 = 3,1. EBITDA (M€), 2025 = 4,6. EBITDA (M€), Évolution = +48,4 %. Résultat net
3 0.016528925 plan_strategique_2026-2028.pptx • Avril 2026 : signature du contrat de distribution aux Pays-Bas
• Juin 2026 : lancement de l'offre de location longue durée
Veltis SA — Comité de direction — confidentiel
4 0.016393442 rapport_annuel_2025.pdf 7. Perspectives 2026
Pour accompagner la croissance, la direction soumettra au conseil d'administration un projet de deuxième site de pr

## Synthèse et autonomie

- **Dense** : proximité de sens (BGE-M3, cosinus).
- **Sparse** : correspondance lexicale (BM25, IDF).
- **Hybrid** : fusion des classements (RRF).
- **`prefetch`** : collecte des candidats *avant* la fusion.
- **`limit=30`** : nombre de candidats par méthode; **`limit=10`** : résultats finaux.
- **`using`** : nom exact du champ vectoriel (`dense` / `sparse`).

**Auto-exercice, sans copier la solution :** changer la question pour Q01, chercher séparément dense / sparse / hybride, relever le rang du passage « Présentez-vous à 8 h 30 », puis refaire pour Q02 et Q03. Vérifier que les réponses restent complètes après découpage à 100 tokens.

**Limites :** ni Qdrant ni le serveur BGE-M3 de la Z4 n’ont été exécutés depuis ce notebook généré. La compatibilité de l’inférence BM25 doit être confirmée sur l’installation locale. Pour la production : ne pas laisser de vraie clé en dur; ajouter filtres, contrôle des sources, et évaluation avant de choisir l’architecture TECHREG.

**Documentation :** [Qdrant — BM25](https://qdrant.tech/documentation/inference/inference-bm25/) · [Qdrant — Hybrid Search](https://qdrant.tech/documentation/search-tuning/hybrid-search/).

In [19]:
# ============================================================
# CELLULE 15 — COMPARAISON 544 vs 100 TOKENS
# RRF = Reciprocal Rank Fusion = Recherche hybride
# ============================================================

# RRF fusionne deux classements :
# 1. DENSE  : recherche sémantique (BGE-M3)
# 2. SPARSE : recherche lexicale (BM25)
#
# k=60 : paramètre qui influence l'importance des rangs.
# weights=[2.0, 1.0] :
#   - Dense  = poids 2
#   - Sparse = poids 1
#
# Les poids suivent l'ordre des deux Prefetch.

RRF_K = 60
RRF_WEIGHTS = [2.0, 1.0]
TOP_K = 5

# Même question pour les deux méthodes de chunking.
# Réutilise "question" et "question_embedded" de la cellule 10.

collections_to_compare = {
    "544 tokens": COLL_544,
    "100 tokens": COLL_100
}

comparison = {}

for profile, collection_name in collections_to_compare.items():

    # Vérifier que la collection existe et contient des données.
    assert qclient.collection_exists(collection_name), (
        f"Collection absente : {collection_name}"
    )

    count = qclient.count(
        collection_name=collection_name,
        exact=True
    ).count

    assert count > 0, f"Collection vide : {collection_name}"

    # Recherche hybride Dense + Sparse.
    results = qclient.query_points(
        collection_name=collection_name,

        prefetch=[
            # 1. Recherche sémantique
            models.Prefetch(
                query=question_embedded,
                using="dense",
                limit=30
            ),

            # 2. Recherche lexicale BM25
            models.Prefetch(
                query=models.Document(
                    text=question,
                    model=SPARSE_MODEL,
                    options=BM25_OPTIONS
                ),
                using="sparse",
                limit=30
            )
        ],

        # 3. Fusion pondérée des deux classements
        query=models.RrfQuery(
            rrf=models.Rrf(
                k=RRF_K,
                weights=RRF_WEIGHTS
            )
        ),

        limit=TOP_K,
        with_payload=True
    )

    comparison[profile] = results.points

    print(f"\n{'=' * 65}")
    print(f"CHUNKING : {profile}")
    print(f"Collection : {collection_name}")
    print(f"Nombre de chunks : {count}")
    print(f"Question : {question}")
    print(f"RRF : k={RRF_K}, weights={RRF_WEIGHTS}")
    print("=" * 65)

    # Afficher les cinq meilleurs passages.
    for rank, point in enumerate(results.points, start=1):

        payload = point.payload or {}

        print(f"\nRANG {rank}")
        print("Score RRF :", round(point.score, 5))
        print("Document  :", payload.get("file_name"))
        print("Chunk ID  :", payload.get("global_index"))
        print("Contenu   :")
        print(payload.get("text", "")[:700])


# ============================================================
# RÉSUMÉ COMPARATIF
# ============================================================

print("\n" + "=" * 65)
print("COMPARAISON DES DEUX MÉTHODES")
print("=" * 65)

for profile, points in comparison.items():
    print(f"\n{profile}")

    for rank, point in enumerate(points, start=1):
        payload = point.payload or {}

        print(
            f"{rank}. {payload.get('file_name')} "
            f"| RRF={point.score:.5f}"
        )

print("\nComparaison terminée.")


CHUNKING : 544 tokens
Collection : veltis_hybrid_544_20261014
Nombre de chunks : 84
Question : Quel dividende sera proposé aux actionnaires ?
RRF : k=60, weights=[2.0, 1.0]

RANG 1
Score RRF : 0.03347
Document  : pv_conseil_administration_2026-03-12_scan.pdf
Chunk ID  : 69
Contenu   :
2. Arrêt des comptes de 1'exercice 2025
Le directeur financier présente le rapport de gestion et les comptes annuels. Après échange, le conseil arrête les comptes à l'unanimité et décide de proposer à l'assemblée générale la distribution d'un dividende de 0,80€ par action.

RANG 2
Score RRF : 0.03101
Document  : rapport_annuel_2025.pdf
Chunk ID  : 83
Contenu   :
7. Perspectives 2026
Pour accompagner la croissance, la direction soumettra au conseil d'administration un projet de deuxième site de production. Les priorités commerciales de 2026 sont l'ouverture du marché néerlandais et le développement d'une offre de location longue durée.
À propos de Veltis - Fondée en 2011, Veltis SA conçoit et fabrique à V

In [20]:
# ============================================================
# CELLULE 16 — COMPARAISON DENSE / SPARSE / HYBRID
#                POUR 544 ET 100 TOKENS
# ============================================================
#
# DENSE  = Recherche sémantique avec BGE-M3 (cosinus)
# SPARSE = Recherche lexicale avec BM25
# HYBRID = Dense + Sparse avec RRF
#
# RRF = Reciprocal Rank Fusion = Fusion des classements
#
# Paramètres de fusion :
# k = 60
# weights = [2.0, 1.0]
#
# Dense reçoit un poids 2.
# Sparse reçoit un poids 1.
#
# Objectif : comparer 6 configurations sur Q01, Q02 et Q03.
# ============================================================

import pandas as pd
import re
import unicodedata
from IPython.display import display

# ------------------------------------------------------------
# 1. PARAMÈTRES
# ------------------------------------------------------------

TOP_K = 5
RRF_K = 60
RRF_WEIGHTS = [2.0, 1.0]

collections = {
    "544 tokens": COLL_544,
    "100 tokens": COLL_100
}

methods = ["Dense", "Sparse", "Hybrid RRF"]

# ------------------------------------------------------------
# 2. QUESTIONS ET RÉPONSES ATTENDUES
# ------------------------------------------------------------

test_questions = {
    "Q01": {
        "question": (
            "Je commence lundi chez Veltis : "
            "à quelle heure et où dois-je me présenter ?"
        ),
        "expected": "Présentez-vous à 8 h 30"
    },

    "Q02": {
        "question": (
            "Pendant combien de temps vais-je être "
            "accompagné par un collègue à mon arrivée ?"
        ),
        "expected": "3 mois"
    },

    "Q03": {
        "question": (
            "Que se passe-t-il si je ne change pas "
            "mon mot de passe provisoire ?"
        ),
        "expected": "le compte est bloqué"
    }
}

# ------------------------------------------------------------
# 3. NORMALISATION DU TEXTE
# ------------------------------------------------------------

def normalize_text(text):
    # Supprime les différences de casse, accents et espaces.
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    return re.sub(r"\s+", " ", text).strip()


# ------------------------------------------------------------
# 4. FONCTION DE RECHERCHE
# ------------------------------------------------------------

def compare_search(collection_name, method, question, vector):

    if method == "Dense":

        result = qclient.query_points(
            collection_name=collection_name,
            query=vector,
            using="dense",
            limit=TOP_K,
            with_payload=True
        )

    elif method == "Sparse":

        result = qclient.query_points(
            collection_name=collection_name,
            query=models.Document(
                text=question,
                model=SPARSE_MODEL,
                options=BM25_OPTIONS
            ),
            using="sparse",
            limit=TOP_K,
            with_payload=True
        )

    elif method == "Hybrid RRF":

        result = qclient.query_points(
            collection_name=collection_name,

            # Recherche des candidats par les deux méthodes.
            prefetch=[
                models.Prefetch(
                    query=vector,
                    using="dense",
                    limit=30
                ),

                models.Prefetch(
                    query=models.Document(
                        text=question,
                        model=SPARSE_MODEL,
                        options=BM25_OPTIONS
                    ),
                    using="sparse",
                    limit=30
                )
            ],

            # RRF = Reciprocal Rank Fusion
            # k = 60 ; Dense x2 ; Sparse x1
            query=models.RrfQuery(
                rrf=models.Rrf(
                    k=RRF_K,
                    weights=RRF_WEIGHTS
                )
            ),

            limit=TOP_K,
            with_payload=True
        )

    return result.points


# ------------------------------------------------------------
# 5. EXÉCUTER LES 18 TESTS
# ------------------------------------------------------------

# 2 tailles x 3 méthodes x 3 questions = 18 recherches.

rows = []
retrieved_passages = {}

# Vérifier les collections avant de commencer.
for label, collection_name in collections.items():
    assert qclient.collection_exists(collection_name), (
        f"Collection absente : {collection_name}"
    )
    assert qclient.count(
        collection_name=collection_name,
        exact=True
    ).count > 0, f"Collection vide : {collection_name}"

for qid, test in test_questions.items():

    question_text = test["question"]
    expected_text = normalize_text(test["expected"])

    # Un seul embedding dense par question.
    response = embedder.embeddings.create(
        model=DENSE_MODEL,
        input=[question_text]
    )

    question_vector = response.data[0].embedding

    for chunk_size, collection_name in collections.items():

        for method in methods:

            points = compare_search(
                collection_name,
                method,
                question_text,
                question_vector
            )

            correct_rank = None
            retrieved_passages[(qid, chunk_size, method)] = points

            # Chercher le rang du passage contenant la réponse.
            for rank, point in enumerate(points, start=1):

                payload = point.payload or {}

                text = normalize_text(
                    payload.get("text", "")
                )

                source = payload.get("file_name", "")

                if (
                    source == "guide_onboarding.pdf"
                    and expected_text in text
                ):
                    correct_rank = rank
                    break

            # Enregistrer les résultats de cette recherche.
            rows.append({
                "Question": qid,
                "Chunking": chunk_size,
                "Méthode": method,
                "Rang correct": correct_rank,
                "Hit@1": int(correct_rank == 1),
                "Hit@5": int(correct_rank is not None),
                "RR@5": (
                    1 / correct_rank
                    if correct_rank is not None
                    else 0
                )
            })


# ------------------------------------------------------------
# 6. TABLEAU DÉTAILLÉ — 18 TESTS
# ------------------------------------------------------------

df = pd.DataFrame(rows)

print("\nTABLEAU 1 — RANG DES RÉPONSES CORRECTES")
display(
    df[
        ["Question", "Chunking", "Méthode",
         "Rang correct", "Hit@1", "Hit@5"]
    ].fillna("-")
)


# ------------------------------------------------------------
# 7. TABLEAU DE SYNTHÈSE — 6 CONFIGURATIONS
# ------------------------------------------------------------

summary = (
    df.groupby(["Chunking", "Méthode"], sort=False)
    .agg(
        Questions=("Question", "count"),
        Hit_1=("Hit@1", "mean"),
        Hit_5=("Hit@5", "mean"),
        MRR_5=("RR@5", "mean")
    )
    .reset_index()
)

summary["Hit@1 (%)"] = (summary["Hit_1"] * 100).round(1)
summary["Hit@5 (%)"] = (summary["Hit_5"] * 100).round(1)
summary["MRR@5"] = summary["MRR_5"].round(3)

print("\nTABLEAU 2 — COMPARAISON DES 6 CONFIGURATIONS")

display(
    summary[
        ["Chunking", "Méthode", "Questions",
         "Hit@1 (%)", "Hit@5 (%)", "MRR@5"]
    ]
)

print("\nComparaison terminée : 18 recherches réalisées.")


TABLEAU 1 — RANG DES RÉPONSES CORRECTES


,Question,Chunking,Méthode,Rang correct,Hit@1,Hit@5
0,Q01,544 tokens,Dense,5.0,0,1
1,Q01,544 tokens,Sparse,-,0,0
2,Q01,544 tokens,Hybrid RRF,-,0,0
3,Q01,100 tokens,Dense,4.0,0,1
4,Q01,100 tokens,Sparse,-,0,0
5,Q01,100 tokens,Hybrid RRF,5.0,0,1
6,Q02,544 tokens,Dense,1.0,1,1
7,Q02,544 tokens,Sparse,-,0,0
8,Q02,544 tokens,Hybrid RRF,-,0,0
9,Q02,100 tokens,Dense,1.0,1,1



TABLEAU 2 — COMPARAISON DES 6 CONFIGURATIONS


,Chunking,Méthode,Questions,Hit@1 (%),Hit@5 (%),MRR@5
0,544 tokens,Dense,3,66.7,100.0,0.733
1,544 tokens,Sparse,3,33.3,33.3,0.333
2,544 tokens,Hybrid RRF,3,33.3,33.3,0.333
3,100 tokens,Dense,3,66.7,100.0,0.750
4,100 tokens,Sparse,3,33.3,33.3,0.333
5,100 tokens,Hybrid RRF,3,33.3,66.7,0.400



Comparaison terminée : 18 recherches réalisées.


In [21]:
# ============================================================
# CELLULE 16a — COMPARAISON DENSE / SPARSE / HYBRID
#                POUR 544 ET 100 TOKENS
# ============================================================
#
# DENSE  = Recherche sémantique avec BGE-M3 (cosinus)
# SPARSE = Recherche lexicale avec BM25
# HYBRID = Dense + Sparse avec RRF
#
# RRF = Reciprocal Rank Fusion = Fusion des classements
#
# Paramètres de fusion :
# k = 60
# weights = [2.0, 1.0]
#
# Dense reçoit un poids 2.
# Sparse reçoit un poids 1.
#
# Objectif : comparer 6 configurations sur Q01, Q02 et Q03.
# ============================================================

import pandas as pd
import re
import unicodedata
from IPython.display import display

# ------------------------------------------------------------
# 1. PARAMÈTRES
# ------------------------------------------------------------

TOP_K = 5
RRF_K = 60
RRF_WEIGHTS = [1.0, 1.0]

collections = {
    "544 tokens": COLL_544,
    "100 tokens": COLL_100
}

methods = ["Dense", "Sparse", "Hybrid RRF"]

# ------------------------------------------------------------
# 2. QUESTIONS ET RÉPONSES ATTENDUES
# ------------------------------------------------------------

test_questions = {
    "Q01": {
        "question": (
            "Je commence lundi chez Veltis : "
            "à quelle heure et où dois-je me présenter ?"
        ),
        "expected": "Présentez-vous à 8 h 30"
    },

    "Q02": {
        "question": (
            "Pendant combien de temps vais-je être "
            "accompagné par un collègue à mon arrivée ?"
        ),
        "expected": "3 mois"
    },

    "Q03": {
        "question": (
            "Que se passe-t-il si je ne change pas "
            "mon mot de passe provisoire ?"
        ),
        "expected": "le compte est bloqué"
    }
}

# ------------------------------------------------------------
# 3. NORMALISATION DU TEXTE
# ------------------------------------------------------------

def normalize_text(text):
    # Supprime les différences de casse, accents et espaces.
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )
    return re.sub(r"\s+", " ", text).strip()


# ------------------------------------------------------------
# 4. FONCTION DE RECHERCHE
# ------------------------------------------------------------

def compare_search(collection_name, method, question, vector):

    if method == "Dense":

        result = qclient.query_points(
            collection_name=collection_name,
            query=vector,
            using="dense",
            limit=TOP_K,
            with_payload=True
        )

    elif method == "Sparse":

        result = qclient.query_points(
            collection_name=collection_name,
            query=models.Document(
                text=question,
                model=SPARSE_MODEL,
                options=BM25_OPTIONS
            ),
            using="sparse",
            limit=TOP_K,
            with_payload=True
        )

    elif method == "Hybrid RRF":

        result = qclient.query_points(
            collection_name=collection_name,

            # Recherche des candidats par les deux méthodes.
            prefetch=[
                models.Prefetch(
                    query=vector,
                    using="dense",
                    limit=30
                ),

                models.Prefetch(
                    query=models.Document(
                        text=question,
                        model=SPARSE_MODEL,
                        options=BM25_OPTIONS
                    ),
                    using="sparse",
                    limit=30
                )
            ],

            # RRF = Reciprocal Rank Fusion
            # k = 60 ; Dense x2 ; Sparse x1
            query=models.RrfQuery(
                rrf=models.Rrf(
                    k=RRF_K,
                    weights=RRF_WEIGHTS
                )
            ),

            limit=TOP_K,
            with_payload=True
        )

    return result.points


# ------------------------------------------------------------
# 5. EXÉCUTER LES 18 TESTS
# ------------------------------------------------------------

# 2 tailles x 3 méthodes x 3 questions = 18 recherches.

rows = []
retrieved_passages = {}

# Vérifier les collections avant de commencer.
for label, collection_name in collections.items():
    assert qclient.collection_exists(collection_name), (
        f"Collection absente : {collection_name}"
    )
    assert qclient.count(
        collection_name=collection_name,
        exact=True
    ).count > 0, f"Collection vide : {collection_name}"

for qid, test in test_questions.items():

    question_text = test["question"]
    expected_text = normalize_text(test["expected"])

    # Un seul embedding dense par question.
    response = embedder.embeddings.create(
        model=DENSE_MODEL,
        input=[question_text]
    )

    question_vector = response.data[0].embedding

    for chunk_size, collection_name in collections.items():

        for method in methods:

            points = compare_search(
                collection_name,
                method,
                question_text,
                question_vector
            )

            correct_rank = None
            retrieved_passages[(qid, chunk_size, method)] = points

            # Chercher le rang du passage contenant la réponse.
            for rank, point in enumerate(points, start=1):

                payload = point.payload or {}

                text = normalize_text(
                    payload.get("text", "")
                )

                source = payload.get("file_name", "")

                if (
                    source == "guide_onboarding.pdf"
                    and expected_text in text
                ):
                    correct_rank = rank
                    break

            # Enregistrer les résultats de cette recherche.
            rows.append({
                "Question": qid,
                "Chunking": chunk_size,
                "Méthode": method,
                "Rang correct": correct_rank,
                "Hit@1": int(correct_rank == 1),
                "Hit@5": int(correct_rank is not None),
                "RR@5": (
                    1 / correct_rank
                    if correct_rank is not None
                    else 0
                )
            })


# ------------------------------------------------------------
# 6. TABLEAU DÉTAILLÉ — 18 TESTS
# ------------------------------------------------------------

df = pd.DataFrame(rows)

print("\nTABLEAU 1 — RANG DES RÉPONSES CORRECTES")
display(
    df[
        ["Question", "Chunking", "Méthode",
         "Rang correct", "Hit@1", "Hit@5"]
    ].fillna("-")
)


# ------------------------------------------------------------
# 7. TABLEAU DE SYNTHÈSE — 6 CONFIGURATIONS
# ------------------------------------------------------------

summary = (
    df.groupby(["Chunking", "Méthode"], sort=False)
    .agg(
        Questions=("Question", "count"),
        Hit_1=("Hit@1", "mean"),
        Hit_5=("Hit@5", "mean"),
        MRR_5=("RR@5", "mean")
    )
    .reset_index()
)

summary["Hit@1 (%)"] = (summary["Hit_1"] * 100).round(1)
summary["Hit@5 (%)"] = (summary["Hit_5"] * 100).round(1)
summary["MRR@5"] = summary["MRR_5"].round(3)

print("\nTABLEAU 2 — COMPARAISON DES 6 CONFIGURATIONS")

display(
    summary[
        ["Chunking", "Méthode", "Questions",
         "Hit@1 (%)", "Hit@5 (%)", "MRR@5"]
    ]
)

print("\nComparaison terminée : 18 recherches réalisées.")


TABLEAU 1 — RANG DES RÉPONSES CORRECTES


,Question,Chunking,Méthode,Rang correct,Hit@1,Hit@5
0,Q01,544 tokens,Dense,5.0,0,1
1,Q01,544 tokens,Sparse,-,0,0
2,Q01,544 tokens,Hybrid RRF,5.0,0,1
3,Q01,100 tokens,Dense,4.0,0,1
4,Q01,100 tokens,Sparse,-,0,0
5,Q01,100 tokens,Hybrid RRF,4.0,0,1
6,Q02,544 tokens,Dense,1.0,1,1
7,Q02,544 tokens,Sparse,-,0,0
8,Q02,544 tokens,Hybrid RRF,3.0,0,1
9,Q02,100 tokens,Dense,1.0,1,1



TABLEAU 2 — COMPARAISON DES 6 CONFIGURATIONS


,Chunking,Méthode,Questions,Hit@1 (%),Hit@5 (%),MRR@5
0,544 tokens,Dense,3,66.7,100.0,0.733
1,544 tokens,Sparse,3,33.3,33.3,0.333
2,544 tokens,Hybrid RRF,3,33.3,100.0,0.511
3,100 tokens,Dense,3,66.7,100.0,0.750
4,100 tokens,Sparse,3,33.3,33.3,0.333
5,100 tokens,Hybrid RRF,3,33.3,100.0,0.583



Comparaison terminée : 18 recherches réalisées.


In [22]:
# ============================================================
# CELL 17 — VERIFY DENSE AND SPARSE VECTOR STORAGE
# ============================================================

for collection_name in [COLL_544, COLL_100]:

    if not qclient.collection_exists(collection_name):
        print(collection_name, "— collection absent")
        continue

    info = qclient.get_collection(collection_name)

    print("\nCOLLECTION:", collection_name)
    print("Dense fields:", list(info.config.params.vectors.keys()))
    print("Sparse fields:", list(info.config.params.sparse_vectors.keys()))

    # Inspect one stored point.
    points, _ = qclient.scroll(
        collection_name=collection_name,
        limit=1,
        with_vectors=True,
        with_payload=True
    )

    if points:
        print("Stored vector fields:", list(points[0].vector.keys()))
    else:
        print("Collection is empty")


COLLECTION: veltis_hybrid_544_20261014
Dense fields: ['dense']
Sparse fields: ['sparse']
Stored vector fields: ['sparse', 'dense']

COLLECTION: veltis_hybrid_100_20261014
Dense fields: ['dense']
Sparse fields: ['sparse']
Stored vector fields: ['dense', 'sparse']
